## 1. First look at the raw data

Load with no interpretation, to find how missing or unknown values are encoded before deciding how to parse them.


In [22]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../data/raw")  # notebook runs from notebooks/

df_peek = pd.read_csv(
    DATA_DIR / "diabetic_data.csv",
    keep_default_na=False,  # keep "NA", "None", "" etc. as raw text
    low_memory=False,       # infer each column's dtype from the whole file, not per chunk
)

print(df_peek.shape)
print(df_peek.info())
df_peek.head()


(101766, 50)
<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   patient_nbr               101766 non-null  int64
 2   race                      101766 non-null  str  
 3   gender                    101766 non-null  str  
 4   age                       101766 non-null  str  
 5   weight                    101766 non-null  str  
 6   admission_type_id         101766 non-null  int64
 7   discharge_disposition_id  101766 non-null  int64
 8   admission_source_id       101766 non-null  int64
 9   time_in_hospital          101766 non-null  int64
 10  payer_code                101766 non-null  str  
 11  medical_specialty         101766 non-null  str  
 12  num_lab_procedures        101766 non-null  int64
 13  num_procedures            101766 non-null  int64
 14  num_medications   

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


### 1.1 Cardinality of the text columns

How many distinct values each text column has. This decides how to inspect it: columns with few
values can be read in full, columns with many values must be scanned.


In [19]:
# Text columns only (pandas 3 gives them the "str" dtype)
text_cols = df_peek.select_dtypes(include="str").columns

# Distinct values per column, smallest first
cardinality = df_peek[text_cols].nunique().sort_values()
cardinality


citoglipton                   1
examide                       1
acetohexamide                 2
troglitazone                  2
diabetesMed                   2
glipizide-metformin           2
tolbutamide                   2
metformin-rosiglitazone       2
metformin-pioglitazone        2
change                        2
glimepiride-pioglitazone      2
tolazamide                    3
gender                        3
readmitted                    3
glyburide-metformin           4
insulin                       4
miglitol                      4
acarbose                      4
rosiglitazone                 4
pioglitazone                  4
glyburide                     4
glimepiride                   4
chlorpropamide                4
nateglinide                   4
repaglinide                   4
metformin                     4
A1Cresult                     4
max_glu_serum                 4
glipizide                     4
race                          6
weight                       10
age     

### 1.2 Values of each text column

Columns with up to 30 distinct values (what fits in one readable line) are listed in full, so no
value can hide. For the others only the rarest values are shown: odd codes and typos live in the tail.

In [20]:
MAX_LISTED = 30  # readability limit: above this, a column is sampled instead of listed

for col, n_unique in cardinality.items():
    counts = df_peek[col].value_counts()  # each value with its frequency, most frequent first
    if n_unique <= MAX_LISTED:
        print(f"{col}: {counts.to_dict()}")
    else:
        # The tail holds the rarest values, where unusual codes show up
        print(f"{col} ({n_unique} unique), rarest: {counts.tail(10).to_dict()}")


citoglipton: {'No': 101766}
examide: {'No': 101766}
acetohexamide: {'No': 101765, 'Steady': 1}
troglitazone: {'No': 101763, 'Steady': 3}
diabetesMed: {'Yes': 78363, 'No': 23403}
glipizide-metformin: {'No': 101753, 'Steady': 13}
tolbutamide: {'No': 101743, 'Steady': 23}
metformin-rosiglitazone: {'No': 101764, 'Steady': 2}
metformin-pioglitazone: {'No': 101765, 'Steady': 1}
change: {'No': 54755, 'Ch': 47011}
glimepiride-pioglitazone: {'No': 101765, 'Steady': 1}
tolazamide: {'No': 101727, 'Steady': 38, 'Up': 1}
gender: {'Female': 54708, 'Male': 47055, 'Unknown/Invalid': 3}
readmitted: {'NO': 54864, '>30': 35545, '<30': 11357}
glyburide-metformin: {'No': 101060, 'Steady': 692, 'Up': 8, 'Down': 6}
insulin: {'No': 47383, 'Steady': 30849, 'Down': 12218, 'Up': 11316}
miglitol: {'No': 101728, 'Steady': 31, 'Down': 5, 'Up': 2}
acarbose: {'No': 101458, 'Steady': 295, 'Up': 10, 'Down': 3}
rosiglitazone: {'No': 95401, 'Steady': 6100, 'Up': 178, 'Down': 87}
pioglitazone: {'No': 94438, 'Steady': 6976

### 1.3 Scan for missing-value placeholders

Missing values are often written as text placeholders instead of empty cells. Every text column is
scanned for the usual ones: empty or punctuation-only strings (`?`, `-`) and words like `NA`, `null`,
`none`, `unknown`. Each hit is a candidate: whether it really means "missing" is decided case by case.


In [23]:
# Matches the whole value: empty/punctuation-only strings, or a common placeholder word
SENTINEL_PATTERN = r"^\W*$|^(na|n/a|nan|null|none|missing|not available|unknown.*)$"

for col in text_cols:
    # Strip spaces and lowercase before matching, so " NULL " and "Null" are caught too
    normalised = df_peek[col].str.strip().str.lower()
    # Count the matching values as they are written in the file
    hits = df_peek.loc[normalised.str.fullmatch(SENTINEL_PATTERN), col].value_counts()
    if not hits.empty:
        print(f"{col}: {hits.to_dict()}")


race: {'?': 2273}
gender: {'Unknown/Invalid': 3}
weight: {'?': 98569}
payer_code: {'?': 40256}
medical_specialty: {'?': 49949}
diag_1: {'?': 21}
diag_2: {'?': 358}
diag_3: {'?': 1423}
max_glu_serum: {'None': 96420}
A1Cresult: {'None': 84748}


### 1.4 Coded IDs: unknown values in `IDS_mapping.csv`

`admission_type_id`, `discharge_disposition_id` and `admission_source_id` are integer codes, so the
text scan above cannot see their placeholders. Their meaning is in `IDS_mapping.csv`: here we find the
codes whose description means "unknown" and count how often each one is actually used in the data.


In [24]:
# The file stacks three code tables, each starting with a "<column>,description" header row
ids_raw = pd.read_csv(
    DATA_DIR / "IDS_mapping.csv",
    header=None,
    names=["code", "description"],
    dtype=str,
    keep_default_na=False,
    skipinitialspace=True,  # strip the space after the comma, e.g. "9, Not Available"
)

# Drop the "," separator lines between the tables
ids_raw = ids_raw[ids_raw["code"] != ""]

# Tag each code with the column it belongs to: take the header name and fill it down
is_header = ids_raw["code"].str.endswith("_id")
ids_raw["column"] = ids_raw["code"].where(is_header).ffill()
ids_map = ids_raw[~is_header].astype({"code": int})[["column", "code", "description"]]

# "Not Mapped" added after reading the file: the generic text pattern did not include it
ID_SENTINEL_PATTERN = r"^(null|not available|not mapped|unknown.*)$"
is_unknown = ids_map["description"].str.strip().str.lower().str.fullmatch(ID_SENTINEL_PATTERN)
unknown_ids = ids_map[is_unknown].copy()

# How often each "unknown" code actually appears in the data
unknown_ids["n_rows"] = [
    (df_peek[col] == code).sum() for col, code in zip(unknown_ids["column"], unknown_ids["code"])
]
unknown_ids


,column,code,description,n_rows
5,admission_type_id,5,Not Available,4785
6,admission_type_id,6,NULL,5291
8,admission_type_id,8,Not Mapped,320
28,discharge_disposition_id,18,NULL,3691
35,discharge_disposition_id,25,Not Mapped,989
36,discharge_disposition_id,26,Unknown/Invalid,0
51,admission_source_id,9,Not Available,125
57,admission_source_id,15,Not Available,0
58,admission_source_id,17,NULL,6781
61,admission_source_id,20,Not Mapped,161


### 1.5 Missing-value inventory

| Kind | Where | Handling |
| --- | --- | --- |
| `?` | `race`, `weight`, `payer_code`, `medical_specialty`, `diag_1/2/3` | read as `NaN`, decided per column in data prep |
| `Unknown/Invalid` | `gender` (3 rows) | rows dropped in scoping |
| Coded "unknown" IDs (`NULL`, `Not Available`, `Not Mapped`, `Unknown/Invalid` in `IDS_mapping.csv`) | `admission_type_id`, `discharge_disposition_id`, `admission_source_id` | merged into one "Unknown" category in data prep |
| `None` | `A1Cresult`, `max_glu_serum` | **not missing**: the test was not measured, kept as a category |


## 2. Load and scope
### 2.1 Load with explicit missing-value rules



In [25]:
df_raw = pd.read_csv(
    DATA_DIR / "diabetic_data.csv",
    na_values=["?"],        # the dataset's only missing-value token
    keep_default_na=False,  # keep "None" (test not measured) as a category
    low_memory=False,
)

# Fails loudly if pandas ever turns "None" into NaN again
assert (df_raw["A1Cresult"] == "None").any()

# Missing values per column, as count and percentage
n_missing = df_raw.isna().sum()
missing = pd.DataFrame({"n_missing": n_missing, "pct": (n_missing / len(df_raw) * 100).round(1)})
missing[missing["n_missing"] > 0].sort_values("n_missing", ascending=False)


,n_missing,pct
weight,98569,96.9
medical_specialty,49949,49.1
payer_code,40256,39.6
race,2273,2.2
diag_3,1423,1.4
diag_2,358,0.4
diag_1,21,0.0


### 2.2 Scoping

Rules from the source paper (Strack et al., 2014), plus one from section 1:

1. **First encounter per patient** (lowest `encounter_id`), so a patient cannot appear in both training and validation data.
2. **Drop encounters ending in death or hospice** (`discharge_disposition_id` 11, 13, 14, 19, 20, 21): these patients cannot be readmitted.
3. **Drop the 3 rows with `gender` = `Unknown/Invalid`** (see 1.5).

Rule 1 is applied before rule 2, which reads "first encounter" literally. The paper reports 69,984 encounters;
applying the rules in the opposite order gives 69,990, so neither order reproduces it exactly.


In [26]:
# Discharged as expired (11, 19, 20, 21) or to hospice (13, 14)
EXCLUDED_DISCHARGE = [11, 13, 14, 19, 20, 21]

# Row count after each rule, to document how much data each one removes
funnel = {"raw": len(df_raw)}

# encounter_id is the only time proxy in the data: lower id = earlier encounter
df = df_raw.sort_values("encounter_id").drop_duplicates(subset="patient_nbr", keep="first")
funnel["first encounter per patient"] = len(df)

df = df[~df["discharge_disposition_id"].isin(EXCLUDED_DISCHARGE)]
funnel["no death / hospice"] = len(df)

df = df[df["gender"] != "Unknown/Invalid"].reset_index(drop=True)
funnel["known gender"] = len(df)

pd.Series(funnel, name="rows").to_frame()


,rows
raw,101766
first encounter per patient,71518
no death / hospice,69973
known gender,69970


#### Check: is `encounter_id` a valid time proxy?

The data has no dates, so "first encounter" relies on lower `encounter_id` = earlier stay. If that holds, for
patients with several encounters every encounter except the last was followed by another stay, so its
`readmitted` should almost never be `NO`, while the last encounter should often be `NO`.


In [27]:
# Raw data, before deduplication: we need all encounters of each patient
enc = df_raw[["patient_nbr", "encounter_id", "readmitted"]].copy()

# Per patient: number of encounters, and whether this one has the highest id (= the last, if ids follow time)
by_patient = enc.groupby("patient_nbr")["encounter_id"]
enc["n_encounters"] = by_patient.transform("size")
enc["is_last"] = enc["encounter_id"] == by_patient.transform("max")

# Patients with at least two encounters: distribution of readmitted (%) for last vs earlier encounters
multi = enc[enc["n_encounters"] >= 2]
print(f"{multi['patient_nbr'].nunique()} patients with 2+ encounters")
(
    pd.crosstab(multi["is_last"], multi["readmitted"], normalize="index")
    .rename(index={False: "not the last", True: "the last"})
    .mul(100)
    .round(1)
)


16773 patients with 2+ encounters


readmitted,<30,>30,NO
is_last,,,
not the last,26.9,71.5,1.6
the last,6.3,24.1,69.6


Earlier encounters are almost never labelled `NO` (1.6%), while most last encounters are (69.6%). If the ids
were unrelated to time, the two rows would look alike: the id order is consistent with time order, so the lowest
`encounter_id` is a reasonable definition of "first encounter".

The 30.4% of last encounters labelled as readmitted are not a contradiction. The readmission was most likely a
stay outside the dataset's inclusion criteria (for example longer than 14 days, or without a diabetes diagnosis),
so it is not among the rows. The `readmitted` label comes with the dataset and its exact definition is not
documented (see Limitations in the README).


### 2.3 Target

Positive class: readmitted **within 30 days** (`<30`). Readmissions after 30 days (`>30`) and no readmission
(`NO`) are both negative, since the 30-day window is what hospitals are penalised on.


In [29]:
# 1 = readmitted within 30 days; ">30" and "NO" both become 0
df["target"] = (df["readmitted"] == "<30").astype(int)

# Drop the source column so the label cannot leak back in as a feature
df = df.drop(columns="readmitted")

print(f"positives: {df['target'].sum()} of {len(df)} ({df['target'].mean():.1%})")


positives: 6277 of 69970 (9.0%)


### 2.4 Diagnosis groups

`diag_1`, `diag_2` and `diag_3` hold ICD-9 codes (primary, secondary and additional diagnosis), with 700+
distinct values each: too many categories for a model. Following the paper, they are grouped into 9 disease
groups. First, the formats the codes come in, so the mapping handles all of them.


In [33]:
DIAG_COLS = ["diag_1", "diag_2", "diag_3"]

# Replace every digit with "9" so codes collapse into their format, e.g. "250.83" -> "999.99", "V57" -> "V99"
for col in DIAG_COLS:
    shapes = df[col].dropna().str.replace(r"\d", "9", regex=True)
    print(f"{col}: {df[col].isna().sum()} NaN | formats of the other codes: {shapes.value_counts().to_dict()}")



diag_1: 10 NaN | formats of the other codes: {'999': 61889, '999.99': 2800, '999.9': 2748, '99': 1278, 'V99': 918, '9': 326, 'E999': 1}
diag_2: 293 NaN | formats of the other codes: {'999': 62187, '999.99': 3718, 'V99': 1217, '999.9': 986, '99': 878, 'E999': 570, '9': 121}
diag_3: 1224 NaN | formats of the other codes: {'999': 60674, 'V99': 2586, '999.99': 2266, '999.9': 1301, 'E999': 929, '99': 869, '9': 121}


#### Are empty diagnosis codes missing or absent?

`number_diagnoses` counts the diagnoses of the encounter. If `diag_2` is empty when there is only one
diagnosis, the code is absent (no secondary diagnosis), not missing.


In [39]:
# Encounters with 1 or 2 diagnoses: are their later diag columns empty?
df.loc[df["number_diagnoses"] <= 2, ["number_diagnoses", *DIAG_COLS]].head(10)


,number_diagnoses,diag_1,diag_2,diag_3
62,1,250.11,NaN,NaN
125,1,250.83,NaN,NaN
181,2,250.02,401,NaN
183,2,493,250,NaN
205,1,250.03,NaN,NaN
238,2,250.02,278,NaN
252,1,250.8,NaN,NaN
272,2,250.13,462,NaN
275,2,824,250.01,NaN
283,2,250.13,276,NaN


In [41]:
# Cap at 3 so all encounters with 3+ diagnoses fall in one row
n_diag = df["number_diagnoses"].clip(upper=3).rename("number_diagnoses (3 = 3+)")

# NaN count per diagnosis column, by number of diagnoses, next to the size of each group
nan_by_group = df[DIAG_COLS].isna().groupby(n_diag).sum()
nan_by_group.insert(0, "n_encounters", n_diag.value_counts().sort_index())
nan_by_group


,n_encounters,diag_1,diag_2,diag_3
number_diagnoses (3 = 3+),,,,
1,193,0,193,193
2,881,0,2,881
3,68896,10,98,150


Every encounter with 1 diagnosis has empty `diag_2` and `diag_3`, and every encounter with 2 diagnoses has an empty
`diag_3`: those codes are **absent** (there is no such diagnosis), not missing. The remaining empty codes
(10 in `diag_1`, 100 in `diag_2`, 150 in `diag_3`) belong to encounters that should have the code: those are
**missing**. The mapping below labels the two cases differently: "Absent" and "Missing".


#### Mapping the codes to groups

ICD-9 codes are mapped to the 9 groups of the paper by numeric range (the codes are compared as numbers, so
stripped leading zeros such as `"8"` for `008` fall correctly into "Other"). `V` and `E` codes go to "Other".
Empty codes become "Absent" or "Missing" as shown above.


In [43]:
def icd9_group(code: str | float) -> str:
    """Map an ICD-9 code to one of the 9 disease groups of Strack et al. (2014)."""
    if pd.isna(code):
        return "Missing"
    if code[0] in "VE":  # supplementary (V) and external-cause (E) codes
        return "Other"
    value = float(code)
    chapter = int(value)  # integer part: "786.05" -> 786
    if 390 <= value < 460 or chapter == 785:
        return "Circulatory"
    if 460 <= value < 520 or chapter == 786:
        return "Respiratory"
    if 520 <= value < 580 or chapter == 787:
        return "Digestive"
    if chapter == 250:
        return "Diabetes"
    if 800 <= value < 1000:
        return "Injury"
    if 710 <= value < 740:
        return "Musculoskeletal"
    if 580 <= value < 630 or chapter == 788:
        return "Genitourinary"
    if 140 <= value < 240:
        return "Neoplasms"
    return "Other"


# One case per code shape found in 2.4: the function must handle them all
assert icd9_group("428") == "Circulatory"
assert icd9_group("250.83") == "Diabetes"
assert icd9_group("786.05") == "Respiratory"
assert icd9_group("8") == "Other"
assert icd9_group("V57") == "Other"
assert icd9_group(float("nan")) == "Missing"

# Empty code beyond number_diagnoses = no such diagnosis -> "Absent" instead of "Missing"
for position, col in enumerate(DIAG_COLS, start=1):
    groups = df[col].map(icd9_group)
    absent = df[col].isna() & (df["number_diagnoses"] < position)
    df[f"{col}_group"] = groups.mask(absent, "Absent")

df[[f"{col}_group" for col in DIAG_COLS]].apply(pd.Series.value_counts)


,diag_1_group,diag_2_group,diag_3_group
Absent,NaN,193,1074
Circulatory,21383.0,22078,20863
Diabetes,5748.0,9700,12546
Digestive,6487.0,2854,2699
Genitourinary,3440.0,5328,4047
Injury,4692.0,1822,1408
Missing,10.0,100,150
Musculoskeletal,4064.0,1295,1368
Neoplasms,2538.0,1599,1145
Other,12122.0,18076,20020
